# Experiment 5: CNN for Image Recognition

## Step 1
Import the required libraries.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import kagglehub

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Step 2
Load the dataset, reshape the images to 1 x 28 x 28 and split it into training, validation and test sets.

In [2]:
def load_mnist():
    path = kagglehub.dataset_download("oddrationale/mnist-in-csv")
    tr = glob.glob(os.path.join(path, "**", "mnist_train.csv"), recursive=True)[0]
    te = glob.glob(os.path.join(path, "**", "mnist_test.csv"), recursive=True)[0]
    a = pd.read_csv(tr).values.astype(np.float32)
    b = pd.read_csv(te).values.astype(np.float32)
    return (
        torch.tensor(a[:, 1:] / 255.0),
        torch.tensor(a[:, 0]).long(),
        torch.tensor(b[:, 1:] / 255.0),
        torch.tensor(b[:, 0]).long(),
    )


def loader(X, y, bs, shuffle):
    return DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=shuffle)


Xtr, ytr, Xte, yte = load_mnist()
Xtr = Xtr.view(-1, 1, 28, 28)
Xte = Xte.view(-1, 1, 28, 28)
Xt, yt = Xtr[:-6000], ytr[:-6000]
Xv, yv = Xtr[-6000:], ytr[-6000:]
train_dl = loader(Xt, yt, 128, True)
val_dl = loader(Xv, yv, 500, False)
test_dl = loader(Xte, yte, 500, False)
print(Xt.shape, Xv.shape, Xte.shape)

torch.Size([54000, 1, 28, 28]) torch.Size([6000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


## Step 3
Define the convolutional layers, pooling layers, flatten layer and fully connected layers.

In [3]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


epochs = 8
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=epochs * len(train_dl))
print(f"parameters: {sum(p.numel() for p in model.parameters())}")

parameters: 468202


## Step 4
Train the CNN for 8 epochs and monitor the validation accuracy.

In [4]:
def evaluate(model, dl):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += len(y)
    return correct / total


def fit(model, dl, epochs, opt, sched=None, val_dl=None, verbose=True):
    for ep in range(epochs):
        model.train()
        tot = 0.0
        n = 0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
            if sched is not None and not isinstance(sched, torch.optim.lr_scheduler.ReduceLROnPlateau):
                sched.step()
            tot += loss.item() * len(y)
            n += len(y)
        msg = f"epoch {ep + 1}/{epochs} loss {tot / n:.4f}"
        if val_dl is not None:
            va = evaluate(model, val_dl)
            msg += f" val_acc {va:.4f}"
            if isinstance(sched, torch.optim.lr_scheduler.ReduceLROnPlateau):
                sched.step(1 - va)
        if verbose:
            print(msg)
    return model


fit(model, train_dl, epochs, opt, sched=sched, val_dl=val_dl)

epoch 1/8 loss 0.4097 val_acc 0.9843
epoch 2/8 loss 0.1010 val_acc 0.9790
epoch 3/8 loss 0.0797 val_acc 0.9910
epoch 4/8 loss 0.0533 val_acc 0.9922
epoch 5/8 loss 0.0389 val_acc 0.9937
epoch 6/8 loss 0.0279 val_acc 0.9942
epoch 7/8 loss 0.0204 val_acc 0.9947
epoch 8/8 loss 0.0162 val_acc 0.9950


CNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=

## Step 5
Test the CNN on the test data, output the overall and per-digit accuracy and save the model.

In [5]:
def per_class_accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = torch.cat([model(X[i:i + 1000].to(device)).argmax(1).cpu() for i in range(0, len(X), 1000)])
    for c in range(10):
        m = y == c
        print(f"class {c}: {(pred[m] == c).float().mean().item():.4f}")


print(f"final test accuracy: {evaluate(model, test_dl):.4f}")
per_class_accuracy(model, Xte, yte)
torch.save(model.state_dict(), "exp05_cnn.pt")

final test accuracy: 0.9957
class 0: 0.9990
class 1: 0.9982
class 2: 0.9971
class 3: 0.9990
class 4: 0.9969
class 5: 0.9944
class 6: 0.9896
class 7: 0.9961
class 8: 0.9969
class 9: 0.9891
